# Hugging Face Yelp Emotion Classification: Selected Business Emotion Dashboard Outputs

This notebook applies the Hugging Face model `j-hartmann/emotion-english-distilroberta-base` to Yelp review text, stores review-level emotion probabilities, and then produces emotion-only outputs for a selected business name.

The seven emotions are:

- anger
- disgust
- fear
- joy
- neutral
- sadness
- surprise




## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Install required packages

In [ ]:
!pip install -q transformers torch accelerate tqdm

## 3. Import libraries

In [ ]:
import os
import gc
import json
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

from tqdm.auto import tqdm
from transformers import pipeline

## 4. Set file paths

In [ ]:
input_path = "/content/drive/MyDrive/yelp_reviews_clean.csv"

review_level_output_path = (
    "/content/drive/MyDrive/yelp_reviews_with_hf_emotions.csv"
)

all_business_summary_output_path = (
    "/content/drive/MyDrive/yelp_all_business_emotion_summary.csv"
)

selected_business_average_output_path = (
    "/content/drive/MyDrive/yelp_selected_business_average_emotions.csv"
)

plot_folder = "/content/drive/MyDrive/yelp_selected_business_emotion_plots"
os.makedirs(plot_folder, exist_ok=True)

print("Input file:", input_path)
print("Review-level output:", review_level_output_path)
print("All-business summary output:", all_business_summary_output_path)
print("Selected-business average output:", selected_business_average_output_path)
print("Plot folder:", plot_folder)

## 5. Define emotion labels

In [ ]:
target_emotions = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
]

## 6. Load the Hugging Face emotion-classification model

In [ ]:
device = 0 if torch.cuda.is_available() else -1

print("Using device:", "GPU" if device == 0 else "CPU")

emotion_classifier = pipeline(
    task="text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    truncation=True,
    device=device
)

## 7. Test the model on one sentence

In [ ]:
test_text = "The food was excellent, but the service was very slow and frustrating."

test_result = emotion_classifier(test_text)

print(test_result)

## 8. Helper functions for emotion extraction

In [ ]:
def normalize_hf_output(prediction):
    """
    Converts Hugging Face pipeline output into a dictionary with all seven emotion scores.

    Expected output shape for one text:
    [
        {'label': 'joy', 'score': 0.71},
        {'label': 'neutral', 'score': 0.12},
        ...
    ]

    Some pipeline versions wrap the result in an extra list, so this function handles both cases.
    """

    if isinstance(prediction, list) and len(prediction) == 1 and isinstance(prediction[0], list):
        prediction = prediction[0]

    scores = {emotion: np.nan for emotion in target_emotions}

    for item in prediction:
        label = item["label"].lower()
        score = float(item["score"])

        if label in scores:
            scores[label] = score

    return scores


def classify_texts_in_batches(texts, batch_size=32):
    """
    Runs the Hugging Face emotion model on a list of texts and returns
    a DataFrame with one row per text and one column per emotion.
    """

    cleaned_texts = [
        str(text).replace("\n", " ").strip() if pd.notna(text) else " "
        for text in texts
    ]

    cleaned_texts = [
        text if text else " "
        for text in cleaned_texts
    ]

    predictions = emotion_classifier(
        cleaned_texts,
        batch_size=batch_size,
        truncation=True
    )

    emotion_rows = [
        normalize_hf_output(prediction)
        for prediction in predictions
    ]

    return pd.DataFrame(emotion_rows)

## 9. Process Yelp reviews and save review-level emotion scores

In [ ]:
chunk_size = 10_000
batch_size = 32

first_chunk = True
total_rows_processed = 0

required_columns = ["business_id", "business_name", "text"]

# Delete existing output so results are not duplicated
if os.path.exists(review_level_output_path):
    os.remove(review_level_output_path)

start_time = time.time()

reader = pd.read_csv(
    input_path,
    usecols=lambda col: col in required_columns,
    chunksize=chunk_size
)

for chunk_number, chunk in enumerate(reader, start=1):

    chunk = chunk.dropna(subset=["business_id", "business_name", "text"]).copy()

    emotion_df = classify_texts_in_batches(
        chunk["text"].tolist(),
        batch_size=batch_size
    )

    for emotion in target_emotions:
        chunk[emotion] = emotion_df[emotion].values

    # Optional compact dictionary column for each review
    chunk["emotion_breakdown"] = chunk[target_emotions].apply(
        lambda row: json.dumps({emotion: float(row[emotion]) for emotion in target_emotions}),
        axis=1
    )

    chunk.to_csv(
        review_level_output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding="utf-8"
    )

    first_chunk = False
    total_rows_processed += len(chunk)

    elapsed_minutes = (time.time() - start_time) / 60

    print(
        f"Chunk {chunk_number}: processed {len(chunk):,} rows; "
        f"total processed {total_rows_processed:,}; "
        f"elapsed {elapsed_minutes:.2f} minutes"
    )

    del chunk, emotion_df
    gc.collect()

print("Review-level emotion classification completed.")
print("Output saved to:", review_level_output_path)


## 10. Load review-level emotion output for aggregation

In [ ]:
emotion_data = pd.read_csv(
    review_level_output_path,
    usecols=lambda col: col in (["business_id", "business_name"] + target_emotions)
)

print("Loaded rows:", len(emotion_data))
display(emotion_data.head())


## 11. Create and save all-business emotion summary by business name


In [ ]:
# Create one business-level emotion summary row per business_name.
# If the same business_name appears across multiple business_id values/locations,
# this combines all matching reviews into one business-name-level summary.

emotion_data["business_name_clean"] = (
    emotion_data["business_name"]
    .astype(str)
    .str.strip()
)

# Remove empty business names, if any
emotion_data = emotion_data[
    emotion_data["business_name_clean"].ne("")
].copy()

# Average emotion probabilities per business name
business_avg = (
    emotion_data
    .groupby("business_name_clean", dropna=False)[target_emotions]
    .mean()
    .reset_index()
)

# Number of reviews per business name
business_review_counts = (
    emotion_data
    .groupby("business_name_clean", dropna=False)
    .size()
    .reset_index(name="review_count")
)

# Number of unique business_id values/locations per business name
business_id_counts = (
    emotion_data
    .groupby("business_name_clean", dropna=False)["business_id"]
    .nunique()
    .reset_index(name="matching_business_id_count")
)

# Combine all business-name-level outputs
business_summary = (
    business_review_counts
    .merge(business_id_counts, on="business_name_clean", how="left")
    .merge(business_avg, on="business_name_clean", how="left")
    .rename(columns={"business_name_clean": "business_name"})
)

business_summary = business_summary[
    ["business_name", "matching_business_id_count", "review_count"] + target_emotions
]

business_summary.to_csv(
    all_business_summary_output_path,
    index=False,
    encoding="utf-8"
)

print("All-business emotion summary saved to:", all_business_summary_output_path)
print("Number of unique business names:", len(business_summary))

display(business_summary.head())



## 12. Select one business by business name


In [ ]:
# Manually set the business name to analyze
selected_business_name = "Starbucks"

# Match the business name case-insensitively and ignore leading/trailing spaces
selected_business_reviews = emotion_data[
    emotion_data["business_name"].astype(str).str.strip().str.lower()
    == selected_business_name.strip().lower()
].copy()

if selected_business_reviews.empty:
    raise ValueError(
        f"No reviews found for business_name = '{selected_business_name}'. "
        "Check the spelling or inspect emotion_data['business_name'].value_counts().head()."
    )

selected_review_count = len(selected_business_reviews)
selected_business_ids = sorted(selected_business_reviews["business_id"].dropna().unique())
selected_location_count = len(selected_business_ids)

print("Selected business name:", selected_business_name)
print("Number of matching business_id values:", selected_location_count)
print("Number of reviews:", selected_review_count)
print("First 10 matching business_id values:", selected_business_ids[:10])

display(selected_business_reviews.head())


## 13. Filter the saved business-level summary and save selected-business average emotion distribution


In [ ]:
# Since the all-business emotion summary was already saved by business_name,
# filter that summary instead of recomputing averages from review-level rows.

business_summary = pd.read_csv(all_business_summary_output_path)

selected_business_summary = business_summary[
    business_summary["business_name"].astype(str).str.strip().str.lower()
    == selected_business_name.strip().lower()
].copy()

if selected_business_summary.empty:
    raise ValueError(
        f"No precomputed business summary found for business_name = '{selected_business_name}'. "
        "Check the spelling or inspect business_summary['business_name'].value_counts().head()."
    )

# There should be one row because the summary is grouped by business_name.
selected_business_summary_row = selected_business_summary.iloc[0]

selected_review_count = int(selected_business_summary_row["review_count"])
selected_location_count = int(selected_business_summary_row["matching_business_id_count"])

selected_average_emotions = (
    selected_business_summary_row[target_emotions]
    .astype(float)
    .sort_values(ascending=False)
)

selected_average_emotions_df = selected_average_emotions.reset_index()
selected_average_emotions_df.columns = ["emotion", "average_score"]

selected_average_emotions_df["business_name"] = selected_business_name
selected_average_emotions_df["review_count"] = selected_review_count
selected_average_emotions_df["matching_business_id_count"] = selected_location_count

# Reorder columns
selected_average_emotions_df = selected_average_emotions_df[
    ["business_name", "matching_business_id_count", "review_count", "emotion", "average_score"]
]

selected_average_emotions_df.to_csv(
    selected_business_average_output_path,
    index=False,
    encoding="utf-8"
)

print("Average emotion distribution for selected business name from precomputed summary:")
display(selected_average_emotions_df)

max_average_emotion = selected_average_emotions.idxmax()
max_average_value = selected_average_emotions.max()

print("Emotion with highest average score:", max_average_emotion)
print("Highest average score:", max_average_value)
print("Saved selected-business average emotion distribution to:", selected_business_average_output_path)



## 14. Plot average emotion distribution for the selected business

In [ ]:
safe_selected_business_name = (
    selected_business_name
    .strip()
    .lower()
    .replace(" ", "_")
    .replace("/", "_")
)

plt.figure(figsize=(8, 5))

plt.bar(
    selected_average_emotions_df["emotion"],
    selected_average_emotions_df["average_score"]
)

plt.title(f"Average Emotion Distribution for {selected_business_name}")
plt.xlabel("Emotion")
plt.ylabel("Average Probability")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.tight_layout()

average_plot_path = os.path.join(
    plot_folder,
    f"{safe_selected_business_name}_average_emotion_distribution.png"
)

plt.savefig(average_plot_path, dpi=300)
plt.show()

print("Saved:", average_plot_path)


## 15. Generate histograms for each emotion for the selected business

In [ ]:
for emotion in target_emotions:
    plt.figure(figsize=(8, 5))

    plt.hist(
        selected_business_reviews[emotion].dropna(),
        bins=50
    )

    plt.title(f"Distribution of {emotion.capitalize()} Scores for {selected_business_name}")
    plt.xlabel(f"{emotion.capitalize()} Probability")
    plt.ylabel("Number of Reviews")
    plt.xlim(0, 1)
    plt.tight_layout()

    plot_path = os.path.join(
        plot_folder,
        f"{safe_selected_business_name}_{emotion}_distribution_hf.png"
    )

    plt.savefig(plot_path, dpi=300)
    plt.show()

    print("Saved:", plot_path)
